# Monitoring and Logging AI Systems — Live Demonstration

**Companion notebook to `monitoring_logging_ai_systems.md`** · Advanced Gen AI programme

This notebook instruments a small customer-support assistant that calls a **real LLM** (Google Gemini, free tier) and sends **real traces** to an observability platform (Langfuse Cloud, free Hobby plan). It then walks through four reproducible scenarios.

### Learning objectives

After running this notebook you should be able to:

1. Explain what a trace, a span, a generation, and a structured log record contain, using real examples.
2. Distinguish **locally measured** latency from **provider-reported** token usage, and both from **illustrative** cost.
3. Recognise three different failure classes (input validation, application bug, provider/configuration error) from their telemetry.
4. Show that a technically successful request can still fail a quality check, and attach that evaluation to the trace.

### What is real and what is not

| Element | Status |
|---|---|
| LLM responses | **Real** inference from the Gemini Developer API |
| Token counts | **Provider-reported** by the API response (where the API returns them) |
| Latency | **Measured locally** in Colab with `time.perf_counter()`; includes network round trip |
| Traces, scores | **Real** telemetry sent to your Langfuse project |
| Knowledge base and retrieval | A small in-memory stand-in for a vector store, so the demo needs no extra infrastructure |
| Failures in Scenario C and D | **Deliberately triggered** by application inputs or configuration, so they are reproducible. The model is never asked to "pretend" to fail. |
| Cost figures | **Illustrative only**, computed from a published paid-tier list price. On the free tier, the actual charge is zero. |

### Architecture

```
┌──────────────── Google Colab ────────────────┐
│  handle_support_request()                     │
│   ├─ validate-input   (span)                  │       HTTPS        ┌───────────────────────┐
│   ├─ retrieve-context (retriever)             │ ─────────────────▶ │ Gemini Developer API  │
│   ├─ build-prompt     (span)                  │                    └───────────────────────┘
│   ├─ llm-generate     (generation) ───────────┤
│   └─ postprocess      (span)                  │   HTTPS (batched)  ┌───────────────────────┐
│  Langfuse SDK (OpenTelemetry-based) ──────────┼──────────────────▶ │ Langfuse Cloud        │
│  Structured JSON logs + pandas summary        │                    │ traces · scores       │
└───────────────────────────────────────────────┘                    └───────────────────────┘
```

### Run order

Run cells **top to bottom**. Approximate live time: 20 minutes. The notebook sends **six** text-generation requests to Gemini, plus one request that is deliberately rejected (Scenario C3).

## Step 0 — One-time setup (before the session)

You need two free accounts. Neither requires a credit card at the time of writing (verified 11 October 2026; re-check before use).

**1. Gemini API key**
- Sign in at [Google AI Studio](https://aistudio.google.com/apikey) and create an API key.
- Check your project's free-tier limits on the [rate-limit page](https://aistudio.google.com/rate-limit).

**2. Langfuse Cloud project**
- Sign up at [cloud.langfuse.com](https://cloud.langfuse.com) (EU region) or [us.cloud.langfuse.com](https://us.cloud.langfuse.com) (US region).
- Create an organisation and a project, then create API keys in the project settings. You receive a **public key** (`pk-lf-...`) and a **secret key** (`sk-lf-...`).

**3. Store the keys as Colab Secrets**
- In Colab, click the **key icon** in the left sidebar.
- Add three secrets with exactly these names and enable **Notebook access** for each:
  - `GEMINI_API_KEY`
  - `LANGFUSE_PUBLIC_KEY`
  - `LANGFUSE_SECRET_KEY`

If secrets are not available, the notebook falls back to environment variables and then to a hidden input prompt.

> **Security consideration —** Never paste keys into code cells. Do not send confidential or personal data through this demo: Google's pricing page states that free-tier Gemini content is used to improve Google's products, and the traces are stored in a third-party service.

## Step 1 — Install dependencies

Only two packages are installed. `pandas` and `matplotlib` are already available in Colab.

In [ ]:
%pip install -q "google-genai>=2.3.0" "langfuse>=4,<5"

## Step 2 — Configuration and credentials

The values below are the **version identifiers** that will be stamped on every trace. In production, these are what make it possible to answer "what changed?" after a regression.

In [ ]:
import os
import getpass

MODEL_ID = "gemini-3.5-flash-lite"   # free-tier model at time of writing; change if unavailable
APP_VERSION = "support-assistant-1.0.0"
PROMPT_VERSION = "refund-prompt-v1"
TEMPERATURE = 0.2
MAX_QUESTION_CHARS = 6000
LANGFUSE_BASE_URL = "https://cloud.langfuse.com"   # EU region; US region: https://us.cloud.langfuse.com


def read_secret(name: str) -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name) or getpass.getpass(f"Enter {name} (input hidden): ")


GEMINI_API_KEY = read_secret("GEMINI_API_KEY")
os.environ["LANGFUSE_PUBLIC_KEY"] = read_secret("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_SECRET_KEY"] = read_secret("LANGFUSE_SECRET_KEY")
os.environ.setdefault("LANGFUSE_BASE_URL", LANGFUSE_BASE_URL)
os.environ["LANGFUSE_TRACING_ENVIRONMENT"] = "workshop-demo"

print("Credentials loaded (values are not displayed).")
print("Model:", MODEL_ID, "| Langfuse:", os.environ["LANGFUSE_BASE_URL"])

## Step 3 — Preflight checks

Before the session, confirm that both services are reachable. This cell does **not** generate any text; it only authenticates with Langfuse and fetches model metadata from Gemini.

Expected output: `langfuse_auth True` and `gemini_model_available True`. If either fails, see the troubleshooting table at the end of the notebook.

In [ ]:
from google import genai
from langfuse import get_client, propagate_attributes

gemini = genai.Client(api_key=GEMINI_API_KEY)
langfuse = get_client()

preflight = {}
try:
    preflight["langfuse_auth"] = langfuse.auth_check()
except Exception as exc:
    preflight["langfuse_auth"] = f"FAILED: {type(exc).__name__}: {str(exc)[:200]}"

INPUT_TOKEN_LIMIT = None
try:
    model_info = gemini.models.get(model=MODEL_ID)
    INPUT_TOKEN_LIMIT = getattr(model_info, "input_token_limit", None)
    preflight["gemini_model_available"] = True
except Exception as exc:
    preflight["gemini_model_available"] = f"FAILED: {type(exc).__name__}: {str(exc)[:200]}"

for name, outcome in preflight.items():
    print(f"{name:24s} {outcome}")
print(f"{'model_input_token_limit':24s} {INPUT_TOKEN_LIMIT}")

## Step 4 — The application being observed

The **Acme Retail Support Assistant** answers policy questions using retrieval-augmented generation (RAG):

1. **Validate** the question (reject empty or oversized input).
2. **Retrieve** relevant policy documents. Here, a keyword router over three in-memory documents stands in for a vector store. The `"broken"` mode reproduces a faulty routing release and is used in Scenario D.
3. **Build** a prompt containing the retrieved policy text.
4. **Generate** an answer with Gemini.
5. **Post-process** the answer using an output template.

The refund policy contains two deliberately specific facts, a **45-day** window and the **Returns Hub**, so that a simple rule-based check can later tell whether an answer used the right document.

In [ ]:
KNOWLEDGE_BASE = {
    "refund-policy": (
        "Refund policy: Customers may request a refund within 45 days of delivery through the "
        "Returns Hub in their Acme account. Approved refunds are issued to the original payment "
        "method within 7 business days. Items marked 'Final Sale' are not refundable."
    ),
    "shipping-policy": (
        "Shipping policy: Standard delivery takes 3-5 business days. Express delivery takes 1-2 "
        "business days and costs $12. Orders placed after 4 p.m. ship the next business day."
    ),
    "warranty-policy": (
        "Warranty policy: Electronics carry a 1-year limited warranty covering manufacturing "
        "defects. Accidental damage is not covered. Claims are filed through the Support Portal "
        "with the order number."
    ),
}

ROUTING_KEYWORDS = {
    "refund-policy": ["refund", "return", "money back"],
    "shipping-policy": ["ship", "deliver", "express"],
    "warranty-policy": ["warranty", "defect", "broken"],
}

SYSTEM_INSTRUCTION = (
    "You are the customer-support assistant for Acme Retail. "
    "Answer ONLY using the policy context provided. "
    "If the context does not contain the answer, say that you do not have that information "
    "and suggest contacting a support agent. Answer in at most 80 words."
)


class InputValidationError(ValueError):
    pass


def validate_question(question: str) -> None:
    if not question or not question.strip():
        raise InputValidationError("Question is empty.")
    if len(question) > MAX_QUESTION_CHARS:
        raise InputValidationError(
            f"Question has {len(question)} characters; the limit is {MAX_QUESTION_CHARS}."
        )


def retrieve_documents(question: str, mode: str = "keyword") -> list:
    if mode == "broken":
        # Reproduces a faulty routing release: every query is sent to the shipping policy.
        return ["shipping-policy"]
    q = question.lower()
    return [doc_id for doc_id, words in ROUTING_KEYWORDS.items() if any(w in q for w in words)]


def build_prompt(question: str, doc_ids: list) -> str:
    context = "\n".join(f"[{d}] {KNOWLEDGE_BASE[d]}" for d in doc_ids) or "(no matching policy found)"
    return f"Policy context:\n{context}\n\nCustomer question:\n{question}"


print("Application components defined:", ", ".join(KNOWLEDGE_BASE))

## Step 5 — Instrumentation

The next cell is the heart of the demonstration. It adds three kinds of telemetry to the application.

**1. A trace with nested observations (Langfuse SDK, built on OpenTelemetry)**

```
support-request              span        root; holds the question and final answer
├─ validate-input            span
├─ retrieve-context          retriever   document IDs returned
├─ build-prompt              span        prompt length
├─ llm-generate              generation  model, parameters, prompt, output, token usage
└─ postprocess               span
```

- `propagate_attributes(...)` stamps every observation with the user, session, application version, prompt version, model, and scenario tag, so traces can be filtered by version later.
- If a step raises an exception, `traced_step` marks that observation with `level="ERROR"` and a status message, and the root span records which stage failed.

**2. Structured JSON logs**

One `request_completed` record per request, printed as JSON (as a container would emit to a log collector) and kept in memory for the summary in Step 6. The **answer text is deliberately not logged**; only lengths, identifiers, and measurements are.

**3. Measurements, labelled by source**

| Field | Source |
|---|---|
| `total_latency_ms`, `llm_latency_ms` | Measured locally in Colab |
| `input_tokens`, `output_tokens`, `thought_tokens`, `total_tokens` | Provider-reported (`interaction.usage`); `None` if the API did not report a value |

> **What this instrumentation does not capture automatically:** provider-side queueing time, time to first token (the demo does not stream), HTTP-level retries inside the SDK, or infrastructure metrics. A production system would add these (see Step 7).

In [ ]:
import json
import time
import uuid
from contextlib import contextmanager
from datetime import datetime, timezone

LOG_RECORDS = []


def log_event(event: str, **fields) -> None:
    record = {"ts": datetime.now(timezone.utc).isoformat(timespec="milliseconds"), "event": event, **fields}
    LOG_RECORDS.append(record)
    print(json.dumps(record, default=str))


@contextmanager
def traced_step(name: str, as_type: str = "span", **kwargs):
    with langfuse.start_as_current_observation(as_type=as_type, name=name, **kwargs) as obs:
        try:
            yield obs
        except Exception as exc:
            obs.update(level="ERROR", status_message=f"{type(exc).__name__}: {exc}"[:500])
            raise


def extract_usage(interaction) -> dict:
    usage = getattr(interaction, "usage", None)

    def read(field):
        value = getattr(usage, field, None) if usage is not None else None
        return int(value) if isinstance(value, (int, float)) else None

    return {
        "input_tokens": read("total_input_tokens"),
        "output_tokens": read("total_output_tokens"),
        "thought_tokens": read("total_thought_tokens"),
        "total_tokens": read("total_tokens"),
    }


def handle_support_request(
    question: str,
    *,
    scenario: str,
    model_id: str = MODEL_ID,
    retrieval_mode: str = "keyword",
    answer_template: str = "{answer}",
    user_id: str = "demo-user-001",
    session_id: str = "workshop-session",
) -> dict:
    request_id = f"req-{uuid.uuid4().hex[:8]}"
    result = {
        "request_id": request_id, "trace_id": None, "scenario": scenario, "status": "ok",
        "failed_stage": None, "error_type": None, "error_message": None, "answer": None,
        "model_id": model_id, "retrieval_mode": retrieval_mode, "retrieved_docs": [],
        "question_chars": len(question or ""), "prompt_chars": None,
        "llm_latency_ms": None, "input_tokens": None, "output_tokens": None,
        "thought_tokens": None, "total_tokens": None,
    }
    stage = "start"
    started = time.perf_counter()

    with langfuse.start_as_current_observation(
        as_type="span", name="support-request", input={"question": question}
    ) as root:
        result["trace_id"] = root.trace_id
        with propagate_attributes(
            trace_name="support-assistant",
            user_id=user_id,
            session_id=session_id,
            version=APP_VERSION,
            tags=[f"scenario-{scenario}"],
            metadata={
                "requestId": request_id,
                "promptVersion": PROMPT_VERSION,
                "modelId": model_id,
                "retrievalMode": retrieval_mode,
            },
        ):
            try:
                stage = "validate-input"
                with traced_step(stage, input={"question_chars": result["question_chars"]}) as obs:
                    validate_question(question)
                    obs.update(output={"valid": True})

                stage = "retrieve-context"
                with traced_step(stage, as_type="retriever",
                                 input={"query": question, "mode": retrieval_mode}) as obs:
                    doc_ids = retrieve_documents(question, retrieval_mode)
                    result["retrieved_docs"] = doc_ids
                    obs.update(output={"doc_ids": doc_ids, "doc_count": len(doc_ids)})

                stage = "build-prompt"
                with traced_step(stage) as obs:
                    prompt = build_prompt(question, doc_ids)
                    result["prompt_chars"] = len(prompt)
                    obs.update(output={"prompt_chars": len(prompt)})

                stage = "llm-generate"
                with traced_step(
                    stage,
                    as_type="generation",
                    model=model_id,
                    model_parameters={"temperature": TEMPERATURE},
                    input=[
                        {"role": "system", "content": SYSTEM_INSTRUCTION},
                        {"role": "user", "content": prompt},
                    ],
                ) as gen:
                    llm_started = time.perf_counter()
                    interaction = gemini.interactions.create(
                        model=model_id,
                        system_instruction=SYSTEM_INSTRUCTION,
                        input=prompt,
                        generation_config={"temperature": TEMPERATURE},
                        store=False,
                    )
                    result["llm_latency_ms"] = round((time.perf_counter() - llm_started) * 1000, 1)
                    answer = interaction.output_text or ""
                    usage = extract_usage(interaction)
                    result.update(usage)
                    reported = {
                        "input": usage["input_tokens"],
                        "output": usage["output_tokens"],
                        "total": usage["total_tokens"],
                    }
                    gen.update(
                        output=answer,
                        usage_details={k: v for k, v in reported.items() if v is not None},
                        metadata={
                            "llmLatencyMsLocal": result["llm_latency_ms"],
                            "thoughtTokens": usage["thought_tokens"],
                            "interactionStatus": str(getattr(interaction, "status", None)),
                        },
                    )

                stage = "postprocess"
                with traced_step(stage) as obs:
                    final_answer = answer_template.format(answer=answer.strip())
                    obs.update(output={"answer_chars": len(final_answer)})

                result["answer"] = final_answer
                root.update(output={"answer": final_answer})

            except Exception as exc:
                result.update(
                    status="error",
                    failed_stage=stage,
                    error_type=type(exc).__name__,
                    error_message=str(exc)[:300],
                )
                root.update(
                    level="ERROR",
                    status_message=f"{stage}: {type(exc).__name__}: {exc}"[:500],
                    output={"error_type": result["error_type"], "failed_stage": stage},
                )

    result["total_latency_ms"] = round((time.perf_counter() - started) * 1000, 1)
    log_event(
        "request_completed",
        app_version=APP_VERSION,
        prompt_version=PROMPT_VERSION,
        **{k: v for k, v in result.items() if k != "answer"},
    )
    return result


def show(result: dict) -> None:
    langfuse.flush()
    try:
        url = langfuse.get_trace_url(trace_id=result["trace_id"])
    except Exception:
        url = None
    print("\n--- Summary ---")
    for key in ("request_id", "status", "failed_stage", "error_type", "error_message",
                "retrieved_docs", "total_latency_ms", "llm_latency_ms",
                "input_tokens", "output_tokens", "thought_tokens", "total_tokens"):
        print(f"{key:17s} {result.get(key)}")
    if result.get("answer"):
        print("\nAnswer (generated by the model):\n" + result["answer"])
    print(f"\nTrace: {url or 'search Langfuse for trace ID ' + str(result['trace_id'])}")


print("Instrumented application ready.")

## Scenario A — A normal request

**Goal:** establish a baseline. Make one real model call, then inspect its telemetry in three places: the structured log line, the summary below it, and the trace in Langfuse.

In [ ]:
QUESTION = "How do I get a refund for my order, and how long do I have to ask for one?"

result_a = handle_support_request(QUESTION, scenario="A-baseline")
show(result_a)

### What to look at in Langfuse

Open the trace link printed above. Traces can take a few seconds to appear; refresh if necessary.

1. **The tree.** Five child observations under `support-request`. Compare the duration of `llm-generate` with the others: in a typical run, model inference dominates the end-to-end time.
2. **The generation.** Select `llm-generate`. Note the model name, the temperature, the system and user messages, the output, and the token usage.
3. **The retriever.** Select `retrieve-context`. Confirm that `refund-policy` was returned.
4. **The attributes.** The trace carries `version`, `promptVersion`, `modelId`, the scenario tag, the user ID, and the session ID.

> **Important —** Local latency (`llm_latency_ms`) includes network time between Colab and Google. It is not the same as the provider's internal processing time.

> **Note on cost in the Langfuse UI —** Langfuse may display a cost for the generation if it recognises the model name and has a price definition for it. That figure is derived from Langfuse's price table, not from an invoice. On the Gemini free tier, the actual charge is zero.

## Scenario B — Longer input, higher consumption

**Goal:** see how input size affects tokens and latency, and why the relationship is not simply proportional.

The customer pastes 25 lines of order history before asking the same question. Note that the word "delivered" in the history also triggers the shipping-policy route, so more context is retrieved: a realistic side effect.

In [ ]:
order_history = "\n".join(
    f"Order #{1000 + i}: item 'Wireless headphones model {i}', delivered on 2026-0{(i % 9) + 1}-1{i % 10}, "
    f"status: delivered, notes: customer called about packaging."
    for i in range(25)
)
LONG_QUESTION = (
    "I have pasted my full order history below so you have all the details.\n"
    + order_history
    + "\n\n" + QUESTION
)

result_b = handle_support_request(LONG_QUESTION, scenario="B-long-input")
show(result_b)

In [ ]:
import pandas as pd
from IPython.display import display

# Paid Standard-tier list prices, USD per 1M tokens, from ai.google.dev/gemini-api/docs/pricing (verified 2026-10-11).
# Output price includes thinking tokens per that page. Free-tier usage is not charged.
ILLUSTRATIVE_PRICES = {"gemini-3.5-flash-lite": {"input": 0.30, "output": 2.50}}


def illustrative_cost_per_1k_requests(r: dict):
    price = ILLUSTRATIVE_PRICES.get(r["model_id"])
    if not price or r["input_tokens"] is None or r["output_tokens"] is None:
        return None
    billed_output = r["output_tokens"] + (r["thought_tokens"] or 0)
    per_request = (r["input_tokens"] * price["input"] + billed_output * price["output"]) / 1_000_000
    return round(per_request * 1000, 4)


comparison = pd.DataFrame([
    {
        "scenario": r["scenario"],
        "question_chars": r["question_chars"],
        "docs_retrieved": len(r["retrieved_docs"]),
        "est_input_tokens (chars/4)": round((r["prompt_chars"] + len(SYSTEM_INSTRUCTION)) / 4) if r["prompt_chars"] else None,
        "input_tokens (reported)": r["input_tokens"],
        "output_tokens (reported)": r["output_tokens"],
        "thought_tokens (reported)": r["thought_tokens"],
        "llm_latency_ms (local)": r["llm_latency_ms"],
        "illustrative USD per 1k requests": illustrative_cost_per_1k_requests(r),
    }
    for r in (result_a, result_b)
]).set_index("scenario")

display(comparison.T)
if INPUT_TOKEN_LIMIT and result_b["input_tokens"]:
    print(f"Context-window utilisation for B: {result_b['input_tokens'] / INPUT_TOKEN_LIMIT:.3%}")

### Interpreting Scenario B

- **Input tokens** grow roughly with input length. The character-based estimate (`chars/4`) is close but not identical to the provider's count; use provider-reported figures for anything that matters.
- **Latency usually grows much less than input tokens.** Output tokens are generated one after another, so output length typically dominates generation time; input is processed in parallel.
- **Output and thinking tokens** depend on the model's behaviour, not just the input size.
- **Cost is asymmetric.** At the illustrative list price, an output token costs about eight times an input token for this model, so a modest change in answer length can matter more than a large change in input.
- **Single measurements are noisy.** Re-running either request will give different latencies. Conclusions about latency require many requests and percentiles, not one comparison.

> **Common Pitfall —** Allowing users (or conversation history) to inflate the prompt without limits. This is why `validate-input` enforces a maximum length, which Scenario C demonstrates.

## Scenario C — Controlled failures

**Goal:** see how three different failure classes appear in telemetry. None of them depends on the model provider misbehaving; each is triggered deterministically.

| Case | Trigger | Failure class |
|---|---|---|
| C1 | A question of about 7,800 characters (limit: 6,000) | **Input validation** — rejected before any model call |
| C2 | An output template that references a missing field | **Application bug after a successful model call** |
| C3 | A model name that does not exist | **Provider / configuration error** — the API rejects the request |

In [ ]:
result_c1 = handle_support_request("Please help me. " * 490, scenario="C1-invalid-input")
show(result_c1)

In [ ]:
result_c2 = handle_support_request(
    QUESTION,
    scenario="C2-postprocess-bug",
    answer_template="{answer}\n\nReference: {ticket_id}",
)
show(result_c2)

In [ ]:
result_c3 = handle_support_request(
    QUESTION,
    scenario="C3-bad-model-config",
    model_id="gemini-model-that-does-not-exist",
)
show(result_c3)

In [ ]:
failures = pd.DataFrame([
    {
        "scenario": r["scenario"],
        "status": r["status"],
        "failed_stage": r["failed_stage"],
        "error_type": r["error_type"],
        "model_called": r["llm_latency_ms"] is not None,
        "tokens_consumed": r["total_tokens"],
        "total_latency_ms": r["total_latency_ms"],
    }
    for r in (result_c1, result_c2, result_c3)
]).set_index("scenario")
display(failures)

### Investigating Scenario C

Open each trace in Langfuse (filter by the tags `scenario-C1-invalid-input`, `scenario-C2-postprocess-bug`, `scenario-C3-bad-model-config`). The observation marked **ERROR** localises the failure immediately.

| | C1 Invalid input | C2 Application bug | C3 Bad model configuration |
|---|---|---|---|
| Failing observation | `validate-input` | `postprocess` | `llm-generate` |
| Model called? | No | **Yes, successfully** | Attempted; rejected by provider |
| Tokens consumed | None | **Yes** — paid for, then discarded | None reported |
| Whose problem? | Expected guardrail behaviour | **Our code** | **Our configuration** (surfaced by the provider) |
| Typical response | Return a clear message to the user | Fix the defect; add a test | Fix configuration; add a deploy-time check |

Key observations:

- **C2 is the most instructive.** The model did its job and consumed tokens, but the user received an error. An error-rate metric alone cannot tell C2 from C3; the trace shows the model call succeeded and the failure is downstream.
- **C3 is a provider error caused by our configuration.** The API response is the evidence, but the fix is on our side. Distinguish this from a genuine provider outage (5xx errors, or 429 rate limiting) by inspecting the error type and message.
- **C1 is not a bug.** Validation rejecting bad input is correct behaviour. Count it separately from genuine errors, or the error rate becomes misleading.

## Scenario D — Technically successful, but wrong

**Goal:** demonstrate that HTTP success does not imply task success, and attach a quality evaluation to the trace.

**Evaluation criterion.** For the refund question, a correct answer must mention both facts from the refund policy:

1. the **45-day** refund window, and
2. the **Returns Hub**.

This is a **deterministic, rule-based check**. It is a basic illustration only: it does not measure tone, completeness, or groundedness, and it would not generalise to other questions. Production systems combine several evaluators (Section 4.5 of the learning module).

**The two runs.** The same question is sent twice:

- **D1** with normal retrieval.
- **D2** with the `"broken"` retrieval mode, which reproduces a faulty routing release that sends every query to the shipping policy.

Both requests should complete with `status = ok`. The question is whether the evaluation can tell them apart.

In [ ]:
import re


def evaluate_refund_answer(answer) -> dict:
    text = (answer or "").lower()
    checks = {
        "mentions_45_day_window": bool(re.search(r"\b45\b|forty[- ]five", text)),
        "mentions_returns_hub": "returns hub" in text,
    }
    return {"passed": all(checks.values()), "checks": checks}


def record_evaluation(result: dict) -> dict:
    evaluation = evaluate_refund_answer(result["answer"])
    langfuse.create_score(
        trace_id=result["trace_id"],
        name="refund_policy_correct",
        value=1.0 if evaluation["passed"] else 0.0,
        data_type="BOOLEAN",
        comment=json.dumps(evaluation["checks"]),
    )
    log_event(
        "evaluation_completed",
        request_id=result["request_id"],
        trace_id=result["trace_id"],
        scenario=result["scenario"],
        evaluator="refund_policy_correct",
        passed=evaluation["passed"],
        **evaluation["checks"],
    )
    return evaluation


result_d1 = handle_support_request(QUESTION, scenario="D1-retrieval-ok")
eval_d1 = record_evaluation(result_d1)
show(result_d1)

In [ ]:
result_d2 = handle_support_request(QUESTION, scenario="D2-retrieval-broken", retrieval_mode="broken")
eval_d2 = record_evaluation(result_d2)
show(result_d2)

In [ ]:
quality = pd.DataFrame([
    {
        "scenario": r["scenario"],
        "status": r["status"],
        "retrieved_docs": ", ".join(r["retrieved_docs"]),
        "total_tokens": r["total_tokens"],
        "llm_latency_ms": r["llm_latency_ms"],
        **e["checks"],
        "evaluation_passed": e["passed"],
    }
    for r, e in ((result_d1, eval_d1), (result_d2, eval_d2))
]).set_index("scenario")
display(quality.T)
langfuse.flush()

### Interpreting Scenario D

- **Both requests succeeded technically.** Same status, similar latency, similar token counts. Every conventional metric says the two are equivalent.
- **Only the evaluation separates them.** D2's answer is usually a polite "I don't have that information," which is *safe* behaviour given the system instruction, but it fails the task.
- **The trace explains why.** In Langfuse, open D2 and select `retrieve-context`: it returned `shipping-policy`. The evaluation score (`refund_policy_correct = false`) is attached to the same trace, so the symptom and the evidence are in one place.
- **Retrieval and answer quality are separate.** D2 is a retrieval failure, not a model failure. Changing the prompt or the model would not fix it.

> **Important —** LLM output is non-deterministic. Occasionally D1 may phrase the answer in a way the rule does not catch (a false negative), or D2 may behave differently from the description above. This is itself a lesson: rule-based checks are brittle, and quality conclusions require evaluation over many requests, not one.

In the Langfuse UI, scores can be filtered and charted over time. In production, the share of sampled requests passing such checks becomes a **quality SLI**.

## Step 6 — From individual requests to metrics

Monitoring works on aggregates. The cell below builds a small dashboard **from the structured log records only**, the same way a log pipeline would compute metrics.

> **Warning —** With fewer than ten requests, percentiles and rates are not statistically meaningful. The purpose here is to show *how* metrics are derived from logs, not to draw conclusions from these numbers.

In [ ]:
import matplotlib.pyplot as plt

requests_df = pd.DataFrame([r for r in LOG_RECORDS if r["event"] == "request_completed"])
evals_df = pd.DataFrame([r for r in LOG_RECORDS if r["event"] == "evaluation_completed"])
for col in ("total_latency_ms", "llm_latency_ms", "input_tokens", "output_tokens", "total_tokens"):
    requests_df[col] = pd.to_numeric(requests_df[col], errors="coerce")

is_validation = requests_df["error_type"] == "InputValidationError"
genuine_errors = (requests_df["status"] == "error") & ~is_validation

summary = pd.Series({
    "requests": len(requests_df),
    "error rate (all errors)": f"{(requests_df['status'] == 'error').mean():.0%}",
    "error rate (excluding validation rejections)": f"{genuine_errors.mean():.0%}",
    "p50 total latency (ms, local)": requests_df["total_latency_ms"].quantile(0.50),
    "p95 total latency (ms, local)": requests_df["total_latency_ms"].quantile(0.95),
    "input tokens (reported, sum)": requests_df["input_tokens"].sum(min_count=1),
    "output tokens (reported, sum)": requests_df["output_tokens"].sum(min_count=1),
    "quality pass rate (evaluated requests)": f"{evals_df['passed'].mean():.0%}" if len(evals_df) else "n/a",
}, name="value")
display(summary.to_frame())

print("Errors by type and stage:")
display(requests_df[requests_df["status"] == "error"].groupby(["failed_stage", "error_type"]).size().to_frame("count"))

colors = ["tab:red" if s == "error" else "tab:blue" for s in requests_df["status"]]
ax = requests_df.plot.bar(x="scenario", y="total_latency_ms", color=colors, legend=False, figsize=(10, 3.5))
ax.set_ylabel("total latency (ms, local)")
ax.set_title("Latency per request (red = error)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Step 7 — Debugging exercise

Use the evidence in this notebook to walk through the six-step method from Section 7 of the learning module.

**Situation:** after a configuration release, the support team reports that customers asking about refunds are being told to "contact a support agent." Dashboards show normal error rate and latency.

| Step | Question | Where the evidence is in this notebook |
|---|---|---|
| 1. Observe | What changed? | Quality pass rate in Step 6; D2 evaluation result |
| 2. Localise | Which component? | D2 trace: `retrieve-context` output |
| 3. Inspect | What happened in affected requests? | Retrieved document IDs; `retrievalMode` attribute on the trace |
| 4. Validate | Can it be reproduced and compared with a baseline? | D1 vs. D2: same question, different retrieval mode |
| 5. Remediate | Smallest justified fix? | Revert the retrieval change (do **not** rewrite the prompt or switch models) |
| 6. Verify | Did it work? | Re-run D with normal retrieval; pass rate returns |

### Exercise: what would a production system add?

Before revealing the suggested answer, list the instrumentation this demo lacks for a real deployment.

<details>
<summary><b>Suggested answer</b></summary>

- **OpenTelemetry-standard instrumentation** across all services, exported through a collector, with trace context propagated from the API gateway.
- **Time to first token**, if responses are streamed.
- **Retry and rate-limit telemetry**: retry counts, 429 responses, backoff time.
- **Retrieval details**: similarity scores, index version, filters, empty-result rate.
- **Version identifiers** for the knowledge-base index and retrieval configuration, in addition to prompt and model.
- **Aggregate metrics** computed on 100% of traffic in a metrics store, with SLO-based burn-rate alerts and named owners.
- **Online evaluation** on a sample of production traffic using validated evaluators (rules, LLM-as-judge, human review), plus an **offline evaluation suite** that gates releases.
- **User feedback and escalation** events linked to trace IDs.
- **Redaction** of personal data before telemetry leaves the trust boundary, **sampling** that keeps all errors, and **retention** limits.
- **Cost reconciliation** against provider billing data, not only list-price estimates.

</details>

## Troubleshooting

| Symptom | Likely cause | Remedy |
|---|---|---|
| `SecretNotFoundError` or a prompt asking for keys | Colab Secret missing, misnamed, or notebook access not enabled | Check the exact names in Step 0 and toggle **Notebook access** |
| `langfuse_auth` is `False` or fails | Wrong keys, or keys from a different region than `LANGFUSE_BASE_URL` | Re-copy keys; set `LANGFUSE_BASE_URL` to the region where the project was created |
| `gemini_model_available` fails with not found | Model name changed or not available to your project | Check the [models page](https://ai.google.dev/gemini-api/docs/models) and set `MODEL_ID` to a current free-tier model |
| `429` / `RESOURCE_EXHAUSTED` | Free-tier rate limit or daily quota reached (limits are per project) | Wait a minute and re-run; check limits in [AI Studio](https://aistudio.google.com/rate-limit); daily quotas reset at midnight Pacific time |
| `5xx` / `service_unavailable` | Provider-side issue | Check the [status page](https://aistudio.google.com/status); retry later |
| `AttributeError` on `interactions` | `google-genai` older than 2.3.0 | Re-run Step 1, then **Runtime → Restart session** and run from Step 2 |
| `AttributeError` on `start_as_current_observation` or `propagate_attributes` | Langfuse SDK older than v4 | Re-run Step 1 and restart the session |
| Trace link opens but trace is missing | Ingestion delay | Wait 10–30 seconds and refresh; ensure `langfuse.flush()` ran |
| Token columns show `None` | The API response did not include that usage field | Expected for some fields (for example, thinking tokens on some models); do not substitute estimates for reported values |
| Variables undefined after a pause | Colab runtime reset | Re-run from Step 1 |

### If a service is unavailable during a live session

- **Model API unavailable:** the concepts can still be taught from traces captured during a successful pre-session rehearsal, provided they are clearly presented as **previously captured** telemetry. Do not present an earlier trace as a new live call.
- **Langfuse unavailable:** the structured logs, summary tables, and Step 6 dashboard still work locally. Telemetry export failures do not stop the application.
- **Both unavailable:** use the learning module's worked examples (Sections 5.2, 7.4, 7.5).

## Cleanup

The next cell sends any buffered telemetry and shuts down the Langfuse client. Afterwards:

- In Langfuse, delete the project (or individual traces) if you do not want to keep the data. On the Hobby plan, data is accessible for 30 days.
- Revoke or rotate the API keys if they were shared or displayed anywhere.
- Remove the Colab Secrets if the notebook will be shared.

In [ ]:
langfuse.flush()
langfuse.shutdown()
del GEMINI_API_KEY
print("Telemetry flushed and Langfuse client shut down. Gemini key removed from memory.")